In [4]:
import os
import shutil
from pathlib import Path

import cv2
import pandas as pd

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass


def _find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").is_dir() and (
            (candidate / "sources").is_dir() or (candidate / ".git").exists()
        ):
            return candidate
    # Respaldo si solo existe 'src' en algún ancestro
    for candidate in [current, *current.parents]:
        if (candidate / "src").is_dir():
            return candidate
    return current


PROJECT_ROOT = _find_project_root(Path.cwd())


def _resolve_dir(env_var: str, default_path: Path) -> Path:
    val = os.getenv(env_var, "").strip()
    if not val:
        return default_path
    p = Path(val)
    return p if p.is_absolute() else (PROJECT_ROOT / p)


RAW_DATA_DIR = _resolve_dir(
    "RAW_DATA_DIR",
    PROJECT_ROOT / "src" / "data" / "raw" / "OneDrive_2026-11-23" / "APO3 PROJECT",
)
PROCESSED_DATA_DIR = _resolve_dir(
    "PROCESSED_DATA_DIR",
    PROJECT_ROOT / "src" / "data" / "processed",
)
SPLIT_DATA_DIR = _resolve_dir(
    "SPLIT_DATA_DIR",
    PROJECT_ROOT / "src" / "data" / "splits",
)

CLASSES = [
    "Bad Quality_Fruits",
    "Regular Quality_Fruits",
    "Good Quality_Fruits",
]
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}
IMG_SIZE = (224, 224)
RANDOM_SEED = 42

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DATA_DIR:", RAW_DATA_DIR)
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)
print("Existe RAW_DATA_DIR:", RAW_DATA_DIR.exists())

PROJECT_ROOT: C:\Users\sgall\Desktop\ProyectoAPOIII
RAW_DATA_DIR: C:\Users\sgall\Desktop\ProyectoAPOIII\src\data\raw\OneDrive_2026-11-23\APO3 PROJECT
PROCESSED_DATA_DIR: C:\Users\sgall\Desktop\ProyectoAPOIII\src\data\processed
Existe RAW_DATA_DIR: True


In [5]:
CLEAN_BEFORE_RUN = True

if CLEAN_BEFORE_RUN:
    for target in (PROCESSED_DATA_DIR, SPLIT_DATA_DIR):
        if target.exists():
            shutil.rmtree(target)
        target.mkdir(parents=True, exist_ok=True)
else:
    PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_DATA_DIR.exists():
    raise FileNotFoundError(f"No existe RAW_DATA_DIR: {RAW_DATA_DIR}")

rows = []
skipped = 0

for class_name in CLASSES:
    class_dir = RAW_DATA_DIR / class_name
    if not class_dir.exists():
        print(f"⚠️ No existe la carpeta de clase: {class_dir}")
        continue

    out_class_dir = PROCESSED_DATA_DIR / class_name
    out_class_dir.mkdir(parents=True, exist_ok=True)

    files = sorted(
        p for p in class_dir.rglob("*")
        if p.is_file()
        and p.suffix.lower() in VALID_EXTENSIONS
        and not p.name.startswith("aug_")
    )

    saved = 0
    for idx, img_path in enumerate(files):
        img = cv2.imread(str(img_path))
        if img is None:
            skipped += 1
            print(f"⚠️ Archivo no legible/corrupto omitido: {img_path}")
            continue

        resized = cv2.resize(img, IMG_SIZE, interpolation=cv2.INTER_AREA)
        fruit_folder = img_path.parent.name if img_path.parent != class_dir else "root"
        prefix = class_name.split()[0].lower()
        out_name = f"{prefix}_{idx:05d}.jpg"
        out_path = out_class_dir / out_name
        cv2.imwrite(str(out_path), resized)

        rows.append(
            {
                "filename": out_name,
                "class": class_name,
                "fruit_subfolder": fruit_folder,
                "source_path": str(img_path),
                "processed_path": str(out_path),
            }
        )
        saved += 1

    print(f"✔ {class_name}: {saved} imágenes procesadas")

df_index = pd.DataFrame(rows)
index_csv = PROCESSED_DATA_DIR / "dataset_index.csv"
df_index.to_csv(index_csv, index=False)

print(f"\nTotal procesadas: {len(df_index)} | Omitidas: {skipped}")
print("Conteo por clase:")
print(df_index["class"].value_counts())

✔ Bad Quality_Fruits: 2235 imágenes procesadas
✔ Regular Quality_Fruits: 4612 imágenes procesadas
✔ Good Quality_Fruits: 2659 imágenes procesadas

Total procesadas: 9506 | Omitidas: 0
Conteo por clase:
class
Regular Quality_Fruits    4612
Good Quality_Fruits       2659
Bad Quality_Fruits        2235
Name: count, dtype: int64
